# CE 310 — Week 11 Assignment Starter Notebook
**Topic:** Hypothesis Testing — t-Tests and ANOVA
**Due:** Wednesday of Week 12 by 9:00 AM — submit to D2L as `.ipynb`

**Instructions:** Set MAJOR, write analysis in blank cells, do not edit ANSWER_ print lines, Run all before submitting.
Rename: `lastname_firstname_week11.ipynb`


## Before You Begin

**File needed:** `Week5_DOE_MedOffice_ZoneTemp_2023.csv`  
This is the same dataset from Weeks 5–7 (Excel) and Weeks 9–10 (Python).

**Upload options:**
1. **Colab Files panel (easiest):** Click the folder icon in the left sidebar → click the upload icon → select the CSV file from your computer.
2. **Google Drive mount:** Click the Drive mount button in the Files panel and navigate to the file.

> If you see `FileNotFoundError`, the CSV is not in Colab's working directory — go back and upload it first.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

MAJOR = "CE"   # <-- change to "ArcE" if Architecture

df = pd.read_csv("Week5_DOE_MedOffice_ZoneTemp_2023.csv")
df['BH'] = (df['BusinessHours'] == 'Yes').astype(int)
print(f"MAJOR = {MAJOR}")


---
## Problem 1 — Hypothesis Testing Extensions (30 pts)
---
### P1a (8 pts) — Nighttime vs. Daytime Cooling


In [ ]:
# Define night/day groups and run t-test + Cohen's d
night = df[(df['Hour'] < 6) | (df['Hour'] >= 22)]['Cooling_kWh']
day   = df[(df['Hour'] >= 6) & (df['Hour'] < 22)]['Cooling_kWh']
# run t-test here, compute pooled std for Cohen's d


In [ ]:
# P1a — ANSWER cells (do not edit)
t_stat_1a, _ = stats.ttest_ind(day, night)
n1, n2 = len(night), len(day)
s1, s2 = night.std(ddof=1), day.std(ddof=1)
pooled_std_1a = np.sqrt(((n1-1)*s1**2 + (n2-1)*s2**2) / (n1+n2-2))
ANSWER_P1a_t = round(t_stat_1a, 4)
ANSWER_P1a_d = round(abs(day.mean() - night.mean()) / pooled_std_1a, 4)
print(f"ANSWER_P1a_t = {ANSWER_P1a_t}")
print(f"ANSWER_P1a_d = {ANSWER_P1a_d}")


*P1a written: Compare Cohen's d to BH-group d (1.30 from lab). Which partition captures occupancy more cleanly?*


---
### P1b (8 pts) — Summer Zone Temperature Compliance


In [ ]:
# Filter to summer occupied hours


In [ ]:
# P1b — ANSWER cells (do not edit)
summer_occ = df[(df['Season'] == 'Summer') & (df['BH'] == 1)]
t_stat_1b, _ = stats.ttest_1samp(summer_occ['Zone_Temp_F'], 75)
ANSWER_P1b_t = round(t_stat_1b, 4)
ANSWER_P1b_frac = round((summer_occ['Zone_Temp_F'] > 75).mean(), 4)
print(f"ANSWER_P1b_t = {ANSWER_P1b_t}")
print(f"ANSWER_P1b_frac = {ANSWER_P1b_frac}")


*P1b written: Explain how small std + large n produces significant t even with negligible mean deviation.*


---
### P1c (7 pts) — Month-Level Post-Hoc Analysis (written/output)


In [ ]:
# P1c — run Tukey HSD and show output
from statsmodels.stats.multicomp import pairwise_tukeyhsd
tukey_month = pairwise_tukeyhsd(df['Cooling_kWh'], df['Month'], alpha=0.05)
print(tukey_month)


*P1c written: Identify the month pair with largest mean difference and one non-significant pair.*


---
### P1d (7 pts) — Effect Size vs. Sample Size (written only)
No code required.


*P1d(a): Which test would fail at n=100? P1d(b): Is the t vs d comparison consistent? P1d(c): Rebuttal.*


---
## Problem 2 — Track Application (18 pts)

---
### Track A — CE: Equipment Sizing Under Uncertainty
*(Skip if MAJOR = "ArcE")*


In [ ]:
# Set up occupied/unoccupied series
occ   = df[df['BH'] == 1]['Cooling_kWh']
unocc = df[df['BH'] == 0]['Cooling_kWh']


In [ ]:
# Track A ANSWER cells (do not edit)
t_occ, p_occ = stats.ttest_1samp(occ, 35)
ANSWER_A2a_t_occ = round(t_occ, 4)
print(f"ANSWER_A2a_t_occ = {ANSWER_A2a_t_occ}")
print(f"  occ mean = {occ.mean():.2f}, p = {p_occ:.4f}")


In [ ]:
t_unocc, p_unocc = stats.ttest_1samp(unocc, 12)
ANSWER_A2b_t_unocc = round(t_unocc, 4)
print(f"ANSWER_A2b_t_unocc = {ANSWER_A2b_t_unocc}")
print(f"  unocc mean = {unocc.mean():.2f}, p = {p_unocc:.4f}")


In [ ]:
ANSWER_A2c_p95 = round(occ.quantile(0.95), 4)
print(f"ANSWER_A2c_p95 = {ANSWER_A2c_p95}")

t_vs_p95, _ = stats.ttest_1samp(occ, ANSWER_A2c_p95)
ANSWER_A2c_t = round(t_vs_p95, 4)
print(f"ANSWER_A2c_t = {ANSWER_A2c_t}")


*Track A written responses.*


---
### Track B — ArcE: ASHRAE 55 Thermal Comfort by Season
*(Skip if MAJOR = "CE")*


In [ ]:
# Set up occupied dataframe


In [ ]:
# Track B ANSWER cells (do not edit)
occ_df = df[df['BH'] == 1]
t_b, p_b = stats.ttest_1samp(occ_df['Zone_Temp_F'], 75)
ANSWER_B2a_t = round(t_b, 4)
ANSWER_B2a_frac = round((occ_df['Zone_Temp_F'] > 75).mean(), 4)
print(f"ANSWER_B2a_t = {ANSWER_B2a_t}")
print(f"ANSWER_B2a_frac = {ANSWER_B2a_frac}")


In [ ]:
# B2b seasonal ANOVA on zone temp (occupied)


In [ ]:
# B2c — Cohen's d for summer vs winter occupied zone temp (do not edit)
summer_z = occ_df[occ_df['Season'] == 'Summer']['Zone_Temp_F']
winter_z = occ_df[occ_df['Season'] == 'Winter']['Zone_Temp_F']
ns, nw = len(summer_z), len(winter_z)
ss, sw = summer_z.std(ddof=1), winter_z.std(ddof=1)
pooled_b = np.sqrt(((ns-1)*ss**2 + (nw-1)*sw**2) / (ns+nw-2))
t_sw, p_sw = stats.ttest_ind(summer_z, winter_z)
ANSWER_B2c_d = round(abs(summer_z.mean() - winter_z.mean()) / pooled_b, 4)
print(f"ANSWER_B2c_d = {ANSWER_B2c_d}")
print(f"  summer mean = {summer_z.mean():.3f}, winter mean = {winter_z.mean():.3f}")
print(f"  t = {t_sw:.4f}, p = {p_sw:.4g}")


*Track B written responses.*


---
## Problem 3 — Written Reflection (12 pts)


*Your P3 reflection here (100–150 words).*
